# Closeout economics and procurement

**Highwood Feeders · simulated, nominal CAD · Phase 2B-1**

Review the modeled margin ledger, diagnose weak lot closeouts, and size purchase and carcass price exposure. The reporting population is **completed placement cohorts**. Placement quarter comes from entry dates. The recorded final exit closes the lot, and every outcome from that lot stays in its cohort. An exit-period view would answer a different business question.

Start with [the Phase 2A checkpoint](../PHASE2A_CHECKPOINT.md), [business-question matrix](../analysis_audit/03_BUSINESS_QUESTION_MATRIX.md) and [KPI definitions](../analysis_audit/04_KPI_DEFINITIONS.md). Snapshotted interpretations and run instructions are in [the findings](../phase2b/01_CLOSEOUT_FINDINGS.md).

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, StrMethodFormatter
import numpy as np
import pandas as pd
from IPython.display import display

from feedlot_analytics import build
from feedlot_analytics.closeout_economics import (
    FINANCIAL_COMPONENTS, MARGIN, REVENUE_AND_COSTS, audit_summary, cohort_economics,
    cost_structure, downside_summary, lot_composition, lot_heterogeneity,
    lot_peer_review, lot_procurement_inputs, price_break_even, procurement_slopes,
    sold_duration_association, sold_performance_slopes,
)
from feedlot_analytics.contracts import FINISHED, MORTALITY
from feedlot_analytics.metrics import aggregate_closeout
from feedlot_analytics.relations import cents

cwd = Path.cwd()
root = next(p for p in (cwd, *cwd.parents) if (p / "PHASE2A_CHECKPOINT.md").is_file())
result = build(root)
checkpoint = json.loads((root / "phase2a" / "reconciliation.json").read_text(encoding="utf-8"))
assert result.reconciliation["summary"] == checkpoint
assert len(result.reconciliation["controls"]) == checkpoint["checks_passed"]
assert all(control["passed"] for control in result.reconciliation["controls"])

tables = result.tables
animals, health, lots = (
    tables[name] for name in ("animal_closeout", "animal_health_summary", "lot_closeout")
)
sold = animals.loc[animals.exit_type.eq(FINISHED)]
overall = aggregate_closeout(animals, health)
by_outcome = aggregate_closeout(animals, health, by=("exit_type",)).set_index("exit_type")
totals = overall.iloc[0]

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlelocation": "left",
    "axes.labelcolor": "#334155", "axes.edgecolor": "#cbd5e1", "font.size": 10,
    "axes.axisbelow": True, "grid.color": "#e2e8f0", "figure.dpi": 120,
})
blue, teal, amber, gray = "#24507c", "#087d78", "#a64912", "#64748b"
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | {checkpoint['checks_passed']} original controls passed")
print(f"{len(animals):,} placed | {len(sold):,} sold | {int(totals.head_mortality):,} deaths")
print(f"{len(lots)} closed lots | {animals.pen_id.nunique()} recorded pens")
print(f"Entries {animals.entry_date.min():%Y-%m-%d} to {animals.entry_date.max():%Y-%m-%d}"
      f" | final exit {animals.exit_date.max():%Y-%m-%d}")

## 1. Audit exceptions and use of the ledger

**Decision.** Use the reconciled closeout ledger for modeled economics. Review each exception at its source grain and escalate gaps that change interpretation.

| Rule | Severity and cause | Reporting implication |
|---|---|---|
| Feed DM and cost rounding | Low. Separate lot-level residuals between independently summed deliveries and closeout allocations. Their magnitudes fit the approved ±0.005 kg DM or CAD per head placed, consistent with source precision. The allocation algorithm is unavailable. | Use allocated feed in the financial ledger and delivered feed for delivery reporting. The two rules can flag the same lot. |
| Untreated encoding | Low. CSV literal `None` versus Excel blank in primary diagnosis and protocol, for 12,900 untreated animals. | The Phase 2A parity rule treats these encodings as equivalent while retaining native source values. |
| Weather source unavailable | Medium. The archived HTTPS proxy attempt blocked the ERA5 acquisition before the provider responded. Coverage is 0 of 704 requested days. | Exclude weather variables from interpretation. Phase 2A retains exposure windows and expected days. |
| Scope of individual breed | High. All 90 lots contain several breeds. Individual breed differs from the dominant label for 1,447 animals. | Use the individual field for biological and carcass comparisons. Lot composition is a separate operational measure. |
| Simulator provenance | High. Simulator code, seed, allocation rules and independent farm records are unavailable. | Use observed ledger arithmetic and descriptive comparisons. Seasonal effects and intervention responses remain unverified. |

The exception unit is one rule, source table and source key. The summary below accounts for all **164** original rows. The maximum absolute lot residuals are **0.09 kg DM** and **CAD 0.10**. Source values and all original exception records are available in the build.

In [ ]:
exceptions = tables["audit_exceptions"]
exception_counts, rounding = audit_summary(exceptions, tables["lot_feed_reconciliation"])
display(exception_counts.rename(columns={"size": "exceptions"}))
display(rounding.round(4))
print(exception_counts.groupby("severity")["size"].sum().to_string())
# Set to True for the full source-key register.
show_audit_register = False
if show_audit_register:
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(exceptions)

## 2. Modeled margin, mortality and cost structure

**Decision.** Use all-outcome CAD per placed head for placement and closeout review. Use all-outcome CAD per sold head for the cost burden carried by marketable output. Use sold-only CAD per sold head to describe survivor economics.

**Calculation and scope.** Booked revenue less purchase, allocated feed, yardage and veterinary costs, summed in exact cents across 15,000 animals in 90 completed lots. Sale revenue is zero for deaths. Sold margin less the recorded mortality burden gives the same all-outcome margin. The mortality column is a subset of the four modeled costs. Direct treatments are already included in veterinary cost. Feed and non-purchase cost of gain divide the corresponding sold costs by gain from the same eligible sold animals.

In [ ]:
ledger = by_outcome.loc[:, ["head_placed", *FINANCIAL_COMPONENTS, "all_outcome_margin_cents"]].copy()
ledger.loc["All outcomes"] = totals[ledger.columns]
ledger["head_placed"] = ledger.head_placed.astype(int)
money_columns = ledger.columns[1:]
ledger[money_columns] = ledger[money_columns] / 100
ledger = ledger.rename(columns=lambda column: column.replace("_cents", "_cad"))
display(ledger.style.format(precision=2, thousands=",", subset=list(ledger.columns[1:])))

kpis = pd.DataFrame(
    [
        ("All outcomes / placed head", totals.head_placed, totals[MARGIN]),
        ("All outcomes / sold head", totals.head_sold, totals.all_outcome_margin_per_head_sold),
        ("Sold only / sold head", totals.head_sold, totals.sold_only_margin_per_head_sold),
        ("Mortality burden / placed head", totals.head_placed, totals.mortality_loss_cad / totals.head_placed),
        ("Mortality burden / death", totals.head_mortality, totals.mortality_loss_cad / totals.head_mortality),
    ],
    columns=["measure", "head_denominator", "CAD per head"],
).set_index("measure")
display(kpis.style.format({"head_denominator": "{:,}", "CAD per head": "{:,.2f}"}))

costs = cost_structure(totals, by_outcome)
display(costs.style.format({
    "all_outcomes_cad": "{:,.2f}", "share_of_modeled_cost": "{:.2%}",
    "cad_per_placed": "{:,.2f}", "mortality_cad": "{:,.2f}",
}))
display(overall[["head_feed_cog_eligible", "feed_cog_gain_kg", "sold_feed_cog",
                 "head_nonpurchase_eligible", "nonpurchase_cog_gain_kg", "sold_nonpurchase_cog"]].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2), layout="constrained")
labels = ["Purchase", "Allocated feed", "Yardage", "Veterinary"]
cost_millions = costs.all_outcomes_cad / 1e6
bars = axes[0].barh(labels, cost_millions, color=blue, height=.57)
axes[0].bar_label(bars, labels=[f"{share:.1%}" for share in costs.share_of_modeled_cost], padding=4)
axes[0].invert_yaxis()
axes[0].set(title="Modeled cost by component", xlabel="CAD million · 15,000 placements", xlim=(0, 34))
axes[0].grid(axis="x")

sold_m, death_m, all_m = totals[["sold_only_margin_cad", "mortality_loss_cad", "all_outcome_margin_cad"]] / 1e6
axes[1].bar([0, 1, 2], [sold_m, death_m, all_m], bottom=[0, all_m, 0],
            color=[blue, amber, teal], width=.6)
for x, top, text in [(0, sold_m, f"{sold_m:.2f}"), (1, sold_m, f"−{death_m:.2f}"), (2, all_m, f"{all_m:.2f}")]:
    axes[1].annotate(text, (x, top), xytext=(0, 4), textcoords="offset points", ha="center")
axes[1].set_xticks([0, 1, 2], ["Sold margin", "240 deaths", "All outcomes"])
axes[1].set(title="Mortality bridge", ylabel="CAD million", ylim=(0, 5.65))
axes[1].grid(axis="y")
plt.show()

**Evidence and action.** Revenue of **CAD 45,710,188.31** and four modeled costs of **CAD 41,328,195.32** yield **CAD 4,381,992.99**, or **CAD 292.13 per placed head**. For the same cohort, all-outcome margin per sold head is CAD 296.88 and sold-only margin per sold head is CAD 331.58. The deaths absorb **CAD 512,194.64**, or CAD 34.15 per placement. Purchase accounts for CAD 468,056.07 of that burden. Purchase and feed together account for 95.15% of booked costs. Prioritize purchase basis and allocated feed review, while maintaining visibility of mortality. Sold feed and non-purchase costs of gain are CAD 2.390 and CAD 2.854 per kg of gain.

**Uncertainty.** These totals are exact for the closed snapshot. The mortality amount measures booked losses. A preventable share, hypothetical sale proceeds and a rate of return on intervention would require independent evidence. The ledger covers four modeled costs. Freight, financing, overhead, settlement adjustments, inventory valuation and cash timing are unavailable.

## 3. Distribution, downside and comparable lot ranking

**Decision.** Route weak closeouts to procurement, feeding or settlement review, and identify where realized losses are concentrated.

**Scope and uncertainty.** The animal tail uses one record per placement and a separate sold-only view. The lot tail uses total CAD from one completed lot per observation. The lot chart and peer rankings use CAD per placed head. The lower-tail cutoff is the empirical fifth percentile with linear interpolation. Count every observation at or below the cutoff, including ties. Count strict negative margins as losses and sum their absolute values before offsets. The observed tails describe one simulated snapshot. Estimating future loss frequency or a capital reserve requires independent history.

In [ ]:
downside = pd.DataFrame({
    "All placements · CAD/head": downside_summary(cents(animals.net_margin_cad)),
    "Sold only · CAD/head": downside_summary(cents(sold.net_margin_cad)),
    "Closed lot · CAD/lot": downside_summary(lots.all_outcome_margin_cents),
}).T
display(downside.style.format({
    "units": "{:,.0f}", "loss_units": "{:,.0f}", "loss_share": "{:.2%}",
    "tail_units": "{:,.0f}", **{field: "{:,.2f}" for field in downside if field.endswith("_cad")},
}))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), layout="constrained")
axes[0].ecdf(animals.net_margin_cad, color=amber, label=f"Placed (n={len(animals):,})")
axes[0].ecdf(sold.net_margin_cad, color=blue, label=f"Sold (n={len(sold):,})")
axes[0].set(title="Animal margin distribution", xlabel="CAD per animal", ylabel="Cumulative share")
axes[0].legend(frameon=False, loc="upper left")
axes[1].ecdf(lots[MARGIN], color=teal)
axes[1].set(title=f"Lot margin distribution (n={len(lots)})", xlabel="All-outcome CAD per placed head")
for axis in axes:
    axis.axvline(0, color=gray, lw=1, ls="--")
    axis.axhline(.05, color=gray, lw=1, ls=":")
    axis.yaxis.set_major_formatter(PercentFormatter(1))
    axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    axis.grid(axis="y")
plt.show()

In [ ]:
peers = lot_peer_review(lots, min_placed=100, min_lots=5)
rank = "rank_all_outcome_per_placed"
quarter_rank = peers.groupby("placement_quarter")[rank]
extremes = peers.loc[peers[rank].eq(1) | peers[rank].eq(quarter_rank.transform("max"))]
peer_fields = [name.replace("_cents", "_contribution_cad_per_placed") for name in REVENUE_AND_COSTS]
peer_display = {
    "eligible_lots_in_quarter": "Quarter lots", "head_placed": "Placed", "head_mortality": "Deaths",
    MARGIN: "CAD/placed", rank: "All-outcome rank", "rank_sold_only_per_sold": "Sold-only rank",
    **dict(zip(peer_fields, ["Revenue Δ", "Purchase Δ", "Feed Δ", "Yardage Δ", "Vet Δ"], strict=True)),
    "all_outcome_margin_contribution_cad_per_placed": "Net Δ",
}
display(extremes.set_index(["placement_quarter", "lot_id"])[list(peer_display)].rename(
    columns=peer_display
).round(2))
print(f"{len(peers)}/{len(lots)} lots eligible | "
      f"{peers[rank].ne(peers.rank_sold_only_per_sold).sum()} change rank under the sold-only KPI")

pen = cohort_economics(animals, health, by=("pen_id",)).set_index("pen_id")
display(pen.nsmallest(5, MARGIN)[["lot_count", "head_placed", "head_mortality", "loss_head",
                                  "loss_head_share", "all_outcome_margin_cad", MARGIN]].round(3))

**Evidence and action.** Among 15,000 placements, **404 lost money**. Gross negative margin was **CAD 522,476.25**, of which deaths contributed 98.0%. The 164 loss-making sold animals incurred CAD 10,281.61. The worst 5% of placements (750 animals) averaged **−CAD 674.82**. All 90 lots and all 40 recorded pen aggregates remained profitable. The weakest lot still made **CAD 16,386.50**.

**Fair comparison.** Rank by all-outcome CAD per placement within the same entry quarter. Require complete head reconciliation, at least 100 placed per lot and at least five eligible lots per quarter. The size rules are reporting choices and all 90 lots qualify here (140–219 head, 8–23 lots per quarter). The bridge compares each lot with the other eligible lots in that quarter using sums divided by peer placements. Revenue contributes positively and higher cost negatively. Peer deltas are CAD per placed head. This reduces timing and size differences, but herd mix, supplier terms and settlement differences remain. The alternate rank uses only sold margins and sold heads.

**Follow-up.** `LOT-2025-027` is last of 23 lots in 2025Q2 at CAD 75.51 per placed head and CAD 215.91 below its peers. Revenue was CAD 513.17 per head below peers, partly offset by CAD 187.13 less purchase cost and CAD 106.38 less feed cost. The lot had seven deaths in 217 placements. Prioritize settlement and marketed volume review alongside mortality. In 2024Q3, `LOT-2024-040` earned higher revenue per placement than its peers but spent more on purchase and feed. `LOT-2024-002` led 2024Q1 at CAD 499.98 per placement. The sold-only convention changes 61 of 90 ranks. Pen summaries are for routing investigation across the assigned lots.

## 4. Procurement region, purchase basis and entry weight

**Decision.** Set placement-quarter benchmarks and identify bids or sourcing patterns worth reviewing. Use actual purchased live kg to price the exposure.

**Calculation and scope.** Compare all outcomes across 90 completed lots and 15,000 placements by procurement region, placement quarter, and their intersection. Each displayed basis is booked purchase cost divided by all placed live kg. The cohort margin and costs divide sums by placed head. The intervals in the chart are the middle 50% of individual lot results, with one observation per lot. The diamond is the head-weighted cohort result. Missing region–quarter cells have no lots. Small cells remain visible with their lot count.

In [ ]:
cohort_fields = {
    "lot_count": "Lots", "head_placed": "Placed", "head_sold": "Sold",
    "purchase_basis_cad_per_kg": "Purchase CAD/kg live", "purchase_cost_cad_per_placed": "Purchase CAD/placed",
    "allocated_feed_cost_cad_per_placed": "Feed CAD/placed", "yardage_cost_cad_per_placed": "Yard CAD/placed",
    "vet_cost_cad_per_placed": "Vet CAD/placed", "production_cost_cad_per_placed": "Cost CAD/placed",
    MARGIN: "Margin CAD/placed", "all_outcome_margin_cad": "Total margin CAD",
    "loss_head_share": "Loss head %", "recorded_morbidity_rate": "Treated %",
}

def cohort_view(frame):
    return frame[list(cohort_fields)].rename(columns=cohort_fields).style.format(
        {"Loss head %": "{:.2%}", "Treated %": "{:.2%}"},
        precision=2, thousands=",",
    )

procurement = lot_procurement_inputs(lots)
dated_lots = lots.merge(procurement[["lot_id", "placement_quarter"]], on="lot_id", validate="one_to_one")
cohort_views = {}
for groups in [("origin_region",), ("placement_quarter",), ("placement_quarter", "origin_region")]:
    view = cohort_economics(animals, health, by=groups).set_index(list(groups))
    cohort_views[groups] = view
    display(cohort_view(view))
cross = cohort_views[("placement_quarter", "origin_region")]
display(cross.lot_count.unstack().fillna(0).astype(int).rename_axis("Lots per placement quarter"))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.5, 5.5), layout="constrained")
for axis_row, dimension in zip(axes, ["origin_region", "placement_quarter"], strict=True):
    profile = lot_heterogeneity(dated_lots, by=(dimension,))
    labels = [f"{group} ({n})" for group, n in profile.lots.items()]
    for axis, prefix, pooled, name in zip(
        axis_row, ["lot_margin", "lot_cost"], [MARGIN, "pooled_cost_cad_per_placed"],
        ["Margin", "Modeled cost"], strict=True,
    ):
        y = np.arange(len(profile))
        axis.hlines(y, profile[f"{prefix}_p25_cad_per_placed"], profile[f"{prefix}_p75_cad_per_placed"],
                    color=blue, lw=3)
        axis.scatter(profile[f"{prefix}_median_cad_per_placed"], y, color=blue, s=24, label="Lot median")
        axis.scatter(profile[pooled], y, marker="D", facecolors="white", edgecolors=teal, s=38,
                     zorder=3, label="Pooled per head")
        axis.set_yticks(y, labels)
        axis.invert_yaxis()
        axis.set(xlabel=f"{name} · CAD per placed head")
        axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
        axis.grid(axis="x")
axes[0, 1].legend(frameon=False, loc="lower right", bbox_to_anchor=(1, 1))
fig.suptitle("Median and middle 50% of lot results · lot count in parentheses", x=.02, ha="left")
plt.show()

**Cohort evidence.** Peace River BC had the highest pooled region margin at CAD 302.65 per placement (14 lots, 2,369 head). Central Saskatchewan returned CAD 285.15 (26 lots, 4,423 head). That CAD 17.50 spread is small relative to the observed spread within regions. Manitoba Foothills had the lowest booked basis (CAD 5.64/kg live) and modeled cost (CAD 2,729.15 per placement). Peace River BC had the highest per-head cost (CAD 2,788.55). Across placement quarters, margin ranged from CAD 258.32 to CAD 341.54 and modeled cost from CAD 2,600.86 to CAD 2,841.21 per placement. The lowest cost occurred in 2024Q4, while the highest margin occurred in 2025Q1. Those measures answer different questions.

**Action and uncertainty.** Use timing-specific comparisons. In 2024Q3, Manitoba led at CAD 289.52 per placement versus Peace River BC at CAD 249.46, each with only two lots. Two region–quarter cells are absent and four contain one lot. Region reflects source geography together with cattle mix, timing and settlement. Supplier identity, freight, transport, shrink and landed cost are unavailable. The lot intervals describe dispersion and provide no sampling interval for a market or supplier ranking.

In [ ]:
bid_association = procurement_slopes(procurement)
display(bid_association.round(2))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.5), layout="constrained", sharey=True)
for quarter, group in procurement.groupby("placement_quarter"):
    axes[0].scatter(group.feeder_purchase_price_per_kg, group[MARGIN], s=25, alpha=.8, label=quarter)
    axes[1].scatter(group.mean_entry_weight_kg, group[MARGIN], s=25, alpha=.8)
axes[0].set(xlabel="Quoted purchase basis · CAD/kg live", ylabel="All-outcome CAD per placed head")
axes[1].set(xlabel="Mean lot entry weight · kg/head placed")
for axis in axes:
    axis.grid()
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncols=6, frameon=False)
fig.suptitle("Purchase-time attributes and final margin · one point per closed lot (n=90)", x=.02, ha="left")
plt.show()

**Purchase price and entry-weight associations.** Fit separate unweighted straight lines at lot grain, pooled and after centering within placement quarter. The price is the lot quote and weight is total entry kg divided by placed head. The sensitivity range refits the quarter-centered slope after withholding one lot at a time. Both fits use 90 lots across six quarters and represent 15,000 placements. The final column further centers within the same quarter and recorded lot sex.

Within quarter, CAD 0.10/kg higher purchase quote was associated with **CAD 20.77 lower margin per placement** (leave-one-lot-out range −23.20 to −18.47). Lots placed 5 kg heavier averaged **CAD 9.26 more margin per placement** along the fitted line (8.35 to 10.25). Centering jointly by quarter and lot sex gave slopes of −CAD 35.16 for price and +CAD 10.20 for weight per stated increment. Mean lot entry weights spanned 306–312 kg for heifers and 346–354 kg for steers. This gap limits comparisons across sex classes. The leave-one-lot-out ranges measure sensitivity to individual lots. Margin directly includes purchase cost, and purchase price and weight can co-vary with carcass volume, health, sex, feed and market conditions. Use a fixed-volume cash scenario for bid exposure and check volume and mix before using weight as a buying criterion.

## 5. Growth, feed conversion and days on feed

**Decision.** Choose metrics for a production review and identify what the snapshot can say about duration. ADG uses gain and inclusive financial DOF. FCR uses allocated dry matter and positive gain. The yard-level ratios use summed numerators and denominators for the same eligible sold animals. The separate animal association uses the recorded individual ratios and booked sold-only margin. Mortality rows remain in the financial ledger and contribute no sold performance measurements.

In [ ]:
metrics = ["head_adg_eligible", "adg_gain_kg", "adg_financial_days", "sold_adg_kg_per_day",
           "head_fcr_eligible", "fcr_dm_kg", "fcr_gain_kg", "sold_fcr"]
display(overall[metrics].round(4))
sold_performance = sold_performance_slopes(animals, draws=2000, seed=20261121)
display(sold_performance.drop(index="financial_dof").round(2))
dof = sold_performance.loc["financial_dof"]
print(f"Lots with variation in sold DOF within the lot: {dof.lots_with_within_variation}/{dof.lots}")
display(sold_duration_association(animals).round(2))

**Evidence and uncertainty.** For ADG and FCR, the slope relates sold margin to one metric at a time after removing lot means for both variables. Whole-lot bootstrap resampling (2,000 draws, seed 20261121) gives descriptive 95% percentile intervals reflecting lot-to-lot sampling sensitivity within this snapshot. The sample is 14,760 eligible sold animals across 90 lots. An extra 0.10 kg/day ADG was associated with **+CAD 70.68 per sold head** (68.85 to 72.39). One additional kg DM/kg gain was associated with **−CAD 395.95 per sold head** (−410.97 to −380.97). Their formulas share gain. The cost ledger uses allocated feed, and carcass revenue is closely linked with weight. These slopes describe overlapping signals. A combined business case would require a model that accounts for that overlap.

Every lot has one common sold DOF, so a within-lot sold DOF slope is unavailable. Comparing the 90 lot means, a 10-day longer duration corresponded to CAD 13.36 higher sold margin per sold head after quarter centering (leave-one-lot-out CAD 7.35 to 16.64). DOF is chosen with target and marketing timing. Use deviations in ADG and FCR as diagnostic flags, and use independent feed, weight and booking records for any decision to change DOF. Feed allocation and survivor selection limit all three comparisons.

## 6. Individual animal groups and lot composition

**Decision.** Review cattle specifications and carcass outcomes, then use operational lot mix to route investigation. Individual breed, sex and frame score come from animal records. Financial comparisons use all outcomes and all placements in the group. HCW, dressing yield, graded share and realized CAD/kg HCW use eligible sold head and the metric-specific denominators. Lot counts show how widely each group is represented and can repeat across the individual categories.

For composition, count actual breed memberships within each complete lot. Compare lot economics by the most represented breed and a descriptive 90% concentration cut. Keep lots and pens as operational units. Sex composition is checked separately because the simulated lot structure limits within-lot comparisons.

In [ ]:
animal_fields = {
    "lot_count": "Lots", "head_placed": "Placed", "head_sold": "Sold", "head_mortality": "Deaths",
    MARGIN: "Margin CAD/placed", "all_outcome_margin_cad": "Total margin CAD",
    "head_hcw_eligible": "HCW head", "mean_hcw_kg": "HCW kg/eligible sold",
    "head_dressing_eligible": "Yield head", "dressing_yield": "Dressing %",
    "head_graded": "Graded head", "prime_aaa_share": "Prime + AAA %",
    "realized_carcass_price_cad_per_kg": "CAD/kg HCW", "loss_head_share": "Loss head %",
}
for group in ("breed", "sex", "frame_score"):
    by_trait = cohort_economics(animals, health, by=(group,)).set_index(group)
    display(by_trait[list(animal_fields)].rename(columns=animal_fields).style.format(
        {"Dressing %": "{:.2%}", "Prime + AAA %": "{:.2%}", "Loss head %": "{:.2%}"},
        precision=2, thousands=",",
    ))

breed_shares = lot_composition(animals, lots)
sex_shares = lot_composition(animals, lots, attribute="sex")
realized_mix = pd.DataFrame({
    "leading_recorded_breed": breed_shares.idxmax(axis=1),
    "leading_breed_share": breed_shares.max(axis=1),
    "breed_share_band": np.where(breed_shares.max(axis=1).ge(.90), "At least 90%", "Below 90%"),
})
composition_lots = lots.merge(realized_mix, on="lot_id", validate="one_to_one")
print(f"Mixed-breed lots: {breed_shares.gt(0).sum(axis=1).gt(1).sum()}/{len(lots)}")
print(f"Single-sex lots: {sex_shares.gt(0).sum(axis=1).eq(1).sum()}/{len(lots)}")
print("Leading breed share range: "
      f"{realized_mix.leading_breed_share.min():.1%} to {realized_mix.leading_breed_share.max():.1%}")
for group in ("leading_recorded_breed", "breed_share_band"):
    lot_mix = lot_heterogeneity(composition_lots, by=(group,))
    display(lot_mix[["lots", "head_placed", "head_sold", MARGIN, "pooled_cost_cad_per_placed",
                     "lot_margin_p25_cad_per_placed", "lot_margin_p75_cad_per_placed"]].round(2))

**Evidence and action.** Individual Charolais Cross averaged CAD 375.87 per placement (3,879 head) and Hereford CAD 159.51 (2,023 head). Black Angus had 86.96% Prime + AAA among 6,372 graded sales but margin of CAD 265.58 per placement. Charolais Cross had higher mean HCW (393.61 vs 378.99 kg) and higher margin despite its lower premium-grade share. Review volume, grade, and purchase basis together. Steer and heifer groups returned CAD 308.94 and CAD 247.07 per placement. All 90 lots were single-sex, so the sex comparison also reflects lot-level decisions. Large frame averaged CAD 312.33 per placement versus CAD 270.86 for medium frame.

Lots with a leading breed share below 90% pooled at CAD 310.23 per placement (41 lots). At or above 90%, the result was CAD 277.19 (49 lots). All lots were mixed and their leading shares ranged from 84.3% to 96.4%. The split is exploratory, and the lot quartiles show overlapping outcomes. Breed, sex, frame, source and date overlap. The group differences describe this simulator. Identifying the effect of changing purchase specifications or mixing policy would require a separate study. A real sourcing trial would require comparable groups and verified settlement and intake data.

## 7. Conditional break-even and price exposure

**Decision.** Size bid and sales-price headroom before setting review thresholds. Use these calculations as fixed-volume ledger scenarios.

- Maximum feeder basis divides booked revenue less allocated feed, yardage and veterinary costs by total placed live kg, including deaths.
- Minimum carcass price divides all modeled costs less any other fixed booked revenue by eligible sold hot carcass kg. The current dataset has carcass revenue on all sales and zero revenue on deaths. Carcass break-even requires positive HCW and full sold HCW coverage.
- Reprice one side at a time. Weights, survival, grades and remaining booked values stay fixed. The carcass equivalent is a single pooled realized CAD/kg HCW for the observed grade mix. Negative feeder headroom means booked basis exceeds the conditional level.

In [ ]:
scenario = price_break_even(overall).rename(index={0: "All closed lots"})
display(scenario.T.style.format("{:,.3f}"))

lot_scenarios = price_break_even(lots.set_index("lot_id"))
scenario_fields = ["purchase_headroom_cad_per_live_kg", "carcass_headroom_cad_per_hcw_kg"]
display(lot_scenarios[scenario_fields].describe().loc[["count", "min", "25%", "50%", "75%", "max"]].round(3))
display(lot_scenarios.nsmallest(5, scenario_fields[0])[
    ["head_placed", "head_sold", "placement_weight_kg", "hcw_kg", *scenario_fields]
].round(3))

**Evidence and action.** The snapshot implies a feeder ceiling of **CAD 6.586/kg live** against CAD 5.724 booked and a carcass floor of **CAD 7.307/kg HCW** against CAD 8.082 realized. A CAD 0.10/kg higher purchase basis reduces the modeled portfolio margin by about **CAD 508,480**. A CAD 0.10/kg lower carcass price reduces it by about **CAD 565,569**. The calculations use 15,000 placements and 14,760 sales with measured HCW. The thinnest headroom across 90 lots was approximately CAD 0.244/kg live and CAD 0.233/kg HCW, both in `LOT-2025-027`. Review lot-specific headroom before portfolio averages. Freight, market responses, premiums by grid component, omitted overhead and working capital need separate inputs before applying the scenario to a live bid or sale.